# Домашняя работа 1. Своя таблица: утечка, пропуски и кодирование

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 1 — Инструменты, данные и первый ориентир |
| Опора | материал семинара 1 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии мы разбирали «Титаник» — учебную таблицу, одинаковую у всех. Дома вы получаете свою собственную: она порождена по вашему ФИО и ни у кого в группе не повторяется. Дефекты в ней те же по природе, что и в «Титанике», но расположены иначе, и найти их придётся самостоятельно.

Три задачи: разведка с поиском утечки, заполнение пропусков и своя реализация One-Hot кодирования. Каждая ячейка — небольшая: почти везде каркас уже написан, дописать нужно одну-две строки.

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/ml_labs/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant, submission_name  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from variants import make_table

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=1)
describe_variant(variant)

print("\nСдавать под именем:", submission_name(STUDENT, lab=1))

In [ ]:
df_raw, meta = make_table(variant)
TASK, TARGET = meta["task"], meta["target"]

print(f"{meta['domain']}: {df_raw.shape[0]} объектов, {df_raw.shape[1] - 1} признаков")
print(f"тип задачи: {TASK}, целевая переменная: {TARGET}")
df_raw.head()

---
# Задача 1. Разведка: что не так с этой таблицей

Повторите на своих данных путь частей 2–4 занятия. Дефекты те же по природе,
что у «Титаника», но есть и два новых: **числовой столбец, записанный строкой**
(типовой экспорт из русского Excel: `'438 900,00'` — пробел как разделитель
разрядов, запятая как десятичный) и **опечатки масштаба** — потерянный
десятичный разделитель, из-за которого отдельные значения завышены примерно
в 100 раз.

### Задание 1.1. Сводка по столбцам

Постройте такую же сводку, как на занятии: тип, число уникальных значений,
доля пропусков. Отдельно найдите столбцы, бесполезные уже сейчас — те, где
одно значение встречается чаще, чем в 99 % строк.

In [ ]:
# TODO (5 строк): сводка по столбцам -- тип, уникальных, доля пропусков.
#   Образец -- часть 2 занятия: pd.DataFrame({...}) из df_raw.dtypes,
#   df_raw.nunique() и df_raw.isna().mean(). Ещё напечатайте число
#   полных дубликатов строк: df_raw.duplicated().sum().

In [ ]:
# TODO (4 строки): для каждого столбца посчитайте долю самого частого значения
#   (df_raw[col].value_counts(normalize=True, dropna=False).iloc[0])
#   и напечатайте те столбцы, где эта доля больше 0.99.

### Задание 1.2. Числовой столбец, записанный строкой

Напишите `to_numeric_safe(s)`: если столбец объектный, но после удаления
пробелов и замены запятой на точку он разбирается в число — преобразовать его.
Если же не разобралась заметная доля непустых значений, столбец действительно
нечисловой (город, категория), и его надо вернуть без изменений.

In [ ]:
def to_numeric_safe(s):
    """Строковый столбец -> числовой, если он на самом деле числовой."""
    if s.dtype != object:
        return s

    # TODO (4 строки):
    #   1) cleaned -- убрать пробелы: .str.replace(r"\s", "", regex=True),
    #      заменить запятую на точку: .str.replace(",", ".", regex=False);
    #   2) converted = pd.to_numeric(cleaned, errors="coerce");
    #   3) lost -- доля значений, которые были непустыми, а числом не стали:
    #      (converted.isna() & s.notna()).mean();
    #   4) вернуть converted, если lost < 0.2, иначе s без изменений.
    raise NotImplementedError

In [ ]:
# Применяем ко всем столбцам и смотрим, что изменилось
df = df_raw.copy()
for col in df.columns:
    before = df[col].dtype
    df[col] = to_numeric_safe(df[col])
    if before != df[col].dtype:
        print(f"{col}: {before} -> {df[col].dtype}")

### Задание 1.3. Категории, дубликаты, вырожденные признаки

В таблице встречаются `Москва`, `москва `, `МОСКВА` — для `pandas` это три
разные категории. Приведите их к единому написанию, а затем удалите полные
дубликаты строк и почти константные столбцы.

Про дубликаты: на занятии мы их **не** удаляли — у «Титаника» нет
идентификатора, и совпадение всех признаков там означало просто двух похожих
пассажиров. Здесь идентификатор есть, и у дублирующихся строк совпадает
**в том числе `id`** — проверьте это сами. Значит, задвоилась запись, а не
объект, и такие строки удаляют.

In [ ]:
print("категорий было:", {c: df[c].nunique() for c in meta["categorical"]})

# TODO (2 строки): для каждого столбца из meta["categorical"] уберите пробелы
#   по краям и приведите к нижнему регистру: .str.strip().str.lower()

print("категорий стало:", {c: df[c].nunique() for c in meta["categorical"]})

In [ ]:
# TODO (5 строк):
#   1) напечатайте число полных дубликатов и удалите их:
#      df.drop_duplicates().reset_index(drop=True);
#   2) соберите список почти константных столбцов (задание 1.1, но по df),
#      удалите их и напечатайте, что осталось.

### Задание 1.4. Опечатки масштаба

Потерянный десятичный разделитель завышает значение примерно в 100 раз. Найдите
такие значения — превышающие 99-й перцентиль столбца более чем в 30 раз, —
поделите их на 100 и напечатайте корреляцию признака с целевой переменной
**до и после** исправления.

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET)

for col in num_cols:
    bad = df[col] > 30 * df[col].quantile(0.99)      # маска подозрительных значений
    if bad.sum() == 0:
        continue
    # TODO (4 строки): запомните корреляцию df[col].corr(df[TARGET]) ДО,
    #   поделите подозрительные значения на 100 (df.loc[bad, col] = ...)
    #   и напечатайте, сколько значений исправлено и какой стала корреляция.

### Задание 1.5. Признак-утечка

На занятии утечкой был `alive` — копия ответа. В вашей таблице она устроена
тоньше: есть столбец `id` — формально технический номер, который не должен
значить ничего.

Проверьте, так ли это, двумя способами без всякой модели: посчитайте корреляцию
`id` с целевой переменной и разбейте объекты на десять групп по возрастанию
`id`, посмотрев среднее значение цели в каждой группе. Если номер и правда
ничего не значит, среднее по группам будет скакать без всякого порядка.

In [ ]:
# TODO (5 строк):
#   1) напечатайте корреляцию id с целью: df["id"].corr(df[TARGET]);
#   2) разбейте объекты на 10 групп по возрастанию id:
#      groups = pd.qcut(df["id"], q=10, labels=False);
#   3) напечатайте среднее значение цели в каждой группе:
#      df.groupby(groups)[TARGET].mean()

In [ ]:
# TODO (5 строк): постройте диаграмму рассеяния id против целевой переменной
#   (ax.scatter, подпишите оси и заголовок), а затем удалите столбец id.

> **Вывод.** Что показали корреляция и средние по группам и откуда взялась эта связь? Чем эта утечка опаснее, чем `alive` с занятия?
>
> *(ваш ответ здесь)*

---
# Задача 2. Чем заполнять пропуски

На занятии мы видели, что `dropna()` может выбросить 80 % выборки и сдвинуть
распределение. Теперь разберёмся, чем заполнять — и почему «медианой» это не
всегда правильный ответ.

Мало посчитать долю пропусков — важно понять, **почему** значение отсутствует:

* **MCAR** — пропуск не зависит ни от чего;
* **MAR** — вероятность пропуска зависит от *других наблюдаемых* признаков
  (доход чаще не указывают клиенты с малым стажем);
* **MNAR** — зависит от самого пропущенного значения (не указывают именно
  большие доходы). Худший случай: по данным его не отличить.

В «Титанике» пропуски в `deck` — типичный MAR: палуба известна в основном для
первого класса. Разница практическая: при MAR заполнение общей медианой
систематически искажает признак. Сейчас измерим, насколько.

### Задание 2.1. Найти столбец с MAR и его драйвер

Для каждого числового столбца с пропусками найдите признак, сильнее всего
связанный с *фактом* пропуска. Мера связи: модуль разности средних в группах
«значение есть» и «значение пропущено», в единицах стандартного отклонения.
Каркас с двойным циклом уже написан — допишите саму меру.

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
with_na = [c for c in num_cols if df[c].isna().any()]
print("столбцы с пропусками:", with_na)

rows = []
for col in with_na:
    is_na = df[col].isna()                 # True там, где значение пропущено
    for other in num_cols:
        if other == col or not df[other].std() > 0:
            continue
        # TODO (2 строки): diff -- модуль разности средних признака other
        #   в группах ~is_na и is_na; добавьте в rows словарь
        #   {"пропуски в": col, "драйвер": other,
        #    "разница средних / sigma": diff / df[other].std()}

# TODO (2 строки): соберите из rows таблицу, отсортируйте по убыванию
#   меры связи и покажите первые пять строк

In [ ]:
# TODO (1 строка): верхняя строка таблицы -- искомая пара
MAR_COL, DRIVER = ..., ...
print(f"сильнее всего связаны: пропуски в «{MAR_COL}» и признак «{DRIVER}»")

### Задание 2.2. Два способа заполнения

Заполните пропуски в `MAR_COL` двумя способами:

* **общей медианой** — одно и то же число всем объектам;
* **медианой внутри квартилей драйвера** — своё число каждой группе.

Эталон, с которым сравниваем, — объекты, где значение известно.

In [ ]:
known = df.loc[df[MAR_COL].notna(), MAR_COL]     # эталон: известные значения
is_na = df[MAR_COL].isna()

# TODO (1 строка): filled_global -- заполнить пропуски общей медианой known.median()

# TODO (3 строки): filled_grouped --
#   groups = pd.qcut(df[DRIVER], q=4, duplicates="drop");
#   group_median = df.groupby(groups, observed=True)[MAR_COL].transform("median");
#   заполнить пропуски group_median, а оставшиеся -- known.median().

In [ ]:
# Сравниваем то, что подставлено вместо пропусков, с эталоном
report = pd.DataFrame({
    "эталон (значение известно)": [known.mean(), known.std()],
    "общая медиана": [filled_global[is_na].mean(), filled_global[is_na].std()],
    "медиана по квартилям": [filled_grouped[is_na].mean(), filled_grouped[is_na].std()],
}, index=["среднее", "ст. отклонение"])
display(report.round(3))

In [ ]:
# TODO (6 строк): постройте на одних осях три гистограммы -- известные значения,
#   заполненные общей медианой и заполненные по квартилям драйвера.
#   Общие корзины: bins = np.histogram_bin_edges(known, bins=30),
#   плотность вместо счётчиков: density=True. Подпишите оси и легенду.

> **Вывод.** Что произошло со стандартным отклонением подставленных значений в каждом случае и чем это опасно? Почему по одному только среднему нельзя решить, какой способ лучше?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 3★. Своя реализация One-Hot кодирования

На занятии мы вызвали `OneHotEncoder` и заметили, что сумма индикаторов по
строке равна единице. Теперь напишем такой кодировщик сами и сверим с
библиотечным численно.

### Задание 3.1. Функция `one_hot`

`one_hot(series, categories=None)` возвращает матрицу индикаторов и список
категорий. Если `categories` задан извне, незнакомые значения кодируются
**нулевой строкой** — так ведёт себя `handle_unknown='ignore'`. Это важно:
на контроле может встретиться категория, которой не было в обучении,
а ширина матрицы обязана сохраниться.

In [ ]:
def one_hot(series, categories=None):
    """Индикаторное кодирование. Возвращает (матрица (n, k), список категорий)."""
    values = pd.Series(series).astype(object)
    if categories is None:
        categories = sorted(values.dropna().unique())     # категории из самих данных
    categories = list(categories)

    # TODO (5 строк):
    #   1) словарь "категория -> номер столбца": {c: j for j, c in enumerate(...)};
    #   2) матрица нулей формы (len(values), len(categories));
    #   3) цикл по объектам: поставить 1.0 в столбец своей категории,
    #      а незнакомое значение и NaN оставить нулевой строкой;
    #   4) вернуть (матрица, categories).
    raise NotImplementedError

### Задание 3.2. Сверка со `scikit-learn`

Разделите известные значения категориального признака пополам. Первую половину
кодируйте «как обучающую» (категории берутся из данных), вторую — «как
контрольную»: категории те же, что были на обучении. Обе матрицы обязаны
совпасть с `OneHotEncoder(handle_unknown='ignore')` численно.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_col = meta["categorical"][0]
values = df[cat_col].dropna()
train, test = values.iloc[: len(values) // 2], values.iloc[len(values) // 2:]

# TODO (6 строк): закодируйте train (категории из данных) и test (категории из
#   train), затем сверьте обе матрицы с OneHotEncoder(handle_unknown="ignore"):
#   enc.fit_transform(train.to_frame()) и enc.transform(test.to_frame()).
#   Напечатайте максимум модуля разности в каждом случае.

In [ ]:
# Проверка поведения на незнакомой категории и на пропуске
strange, _ = one_hot(pd.Series(["такой-категории-нет", np.nan]), categories=cats)

print("незнакомое значение и NaN дают нулевые строки:", strange.sum() == 0)
print("ширина матрицы сохранена:", strange.shape[1] == len(cats))

> **Вывод.** Почему нулевая строка — правильное поведение для незнакомой категории? И чем плоха сумма индикаторов, тождественно равная единице?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Коллега заполнил пропуски средним по всей выборке — включая контрольную часть. Какие два разных дефекта он допустил одновременно?
2. В обучающей выборке признак принимает 40 значений, в контрольной встретилось 41-е. Что произойдёт при `handle_unknown='ignore'` и чем это лучше падения с ошибкой?
3. Сравните утечку `alive` с занятия и утечку `id` из вашей таблицы: какая опаснее на практике и почему?

---

## Обратная связь

Это не оценивается и на балл не влияет — нужно, чтобы поправить работу к
следующему году. Отвечайте одной строкой, честно.

| | |
|---|---|
| Сколько часов заняло | |
| Сложность от 1 до 5 | |
| Что осталось непонятным | |
| Какое задание показалось лишним | |

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.

Имя файла — то, что напечатала ячейка с вариантом: `hw01_Ivanov_I_I.ipynb`.
Номер работы впереди, фамилия и инициалы латиницей. Если сдаёте исправленную
версию, допишите `_v2`.